In [1]:
using CSV
using DataFrames
using Statistics
using Random

using Plots
using Plots.PlotMeasures
using StatsPlots, KernelDensity




In [2]:
include("../src/fxio.jl")
include("../src/entropy.jl")
include("../src/foldexity.jl")

In [87]:
fsize = 4
cutoff = 2
pdbfile = "../testpdb/d1914a1.pdb"

pdb = readpdb_calpha(pdbfile)
xyzcoords = pdb2xyz(pdb)
xyz4mers = coords2kmers(xyzcoords, fsize)

pdbmatrix = pdb2pdbmatrix(pdb)
pdbfragments = coords2kmers(pdbmatrix, fsize)

outdir =  split(basename(pdbfile), ".")[1]
if isdir(outdir)
    rm(outdir, force = true, recursive=true)
end

mkpath(outdir)
pdb = readpdb_calpha(pdbfile)
pdbfragments = coords2kmers(pdbmatrix, 4)
i = 1
for (cl, frag) in zip(results, pdbfragments)
    writepdb(pdbmatrix2pdb(frag), "$outdir/cl$(cl)frag$i.pdb")
    i+=1
end

In [95]:
nfrags = length(xyz4mers)  # Change this to the desired size
matrix = zeros(Float64, nfrags, nfrags)

for i = 1:nfrags # Fill the upper triangle
    for j = i+1:nfrags  # Ensure j >= i for the upper triangle
        rmsd_ij = rmsd(xyz4mers[i], xyz4mers[j])
        matrix[i, j] = rmsd_ij
    end
end

matrix += matrix' 

aver_rmsd = sum(matrix) / (nfrags * nfrags)

2.4552632288453022

In [91]:
cl = hclust(matrix, linkage=:complete)
results = cutree(cl, h=0.5) 
nclusts = length(unique(results))

25